In [1]:
# Importations
import os
import uuid
import psycopg
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType, FloatType, DoubleType
)
from pyspark.sql.functions import (
    col, to_timestamp, concat_ws, lit, current_timestamp, regexp_replace, to_date, date_format
)


In [2]:
# Chargement des variables d'environnement (.env)
load_dotenv()


True

In [3]:
# Paramètres de connexion et chemins
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DATASETS_DIR = os.getenv("DATASETS_DIR")

JDBC_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
JDBC_PROPS = {
    "user": DB_USER,
    "password": DB_PASSWORD,
    "driver": "org.postgresql.Driver"
}


In [4]:
# Choix du fichier CSV à ingérer
csv_filename = "20240616_wind_power_data.csv"

if DATASETS_DIR is None:
	raise ValueError("DATASETS_DIR est None. Vérifiez le .env et load_dotenv().")

csv_path = os.path.join(DATASETS_DIR, csv_filename)


In [5]:
# Initialisation de la session Spark
spark = SparkSession.builder \
    .appName("WindPowerBronzeIngestion") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY") \
    .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 21:45:57 WARN Utils: Your hostname, harpar, resolves to a loopback address: 127.0.1.1; using 192.168.0.2 instead (on interface enp0s31f6)
26/05/03 21:45:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/harpar/DEV/DataEngineering/FinalELTProject/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harpar/.ivy2.5.2/cache
The jars for the packages stored in: /home/harpar/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-6e97fe84-00d2-44ee-9d4f-fbb594dbf825;1.0
	confs: [default]
	found org.postgresql#postgresql;42.6.0 in central
	found org.checkerframework#checker-qual;3.31.0 in central
:: resolution report :: resolve 88ms :: artifacts dl 5ms
	:: modules in use:
	org.chec

In [6]:
# Définition du schéma du CSV
csv_schema = StructType([
    StructField("production_id", IntegerType(), True),
    StructField("date", StringType(), True),
    StructField("time", StringType(), True),
    StructField("turbine_name", StringType(), True),
    StructField("capacity", IntegerType(), True),
    StructField("location_name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("region", StringType(), True),
    StructField("status", StringType(), True),
    StructField("responsible_department", StringType(), True),
    StructField("wind_speed", FloatType(), True),
    StructField("wind_direction", StringType(), True),
    StructField("energy_produced", FloatType(), True)
])


In [7]:
# Lecture du CSV dans un DataFrame Spark
df_raw = spark.read \
    .option("header", "true") \
    .schema(csv_schema) \
    .option("mode", "PERMISSIVE") \
    .option("columnNameOfCorruptRecord", "_corrupt_record") \
    .csv(csv_path)
df_raw.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|status|responsible_department|wind_speed|wind_direction|energy_produced|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+
|         6481|2024-06-16|00-00-00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|Online|            Operations|  17.49365|             W|      1878.9614|
|         6482|2024-06-16|00-00-00|   Turbine B|    2000|   Location 2| 36.7783|-119.4179|Region B|Online|            Operations|  13.09261|             S|     1016.30426|
|         6483|2024-06-16|00-00-00|   Turbine C|    2500|   Location 3| 40.7128|  -74.006|Region C|Online|            Operations|   8.26528|

In [8]:
# Ajout des colonnes de métadonnées et transformation
df_bronze = (
    df_raw
    .withColumn(
        "measured_at",
        to_timestamp(
            concat_ws(" ", col("date"), regexp_replace(col("time"), lit("-"), lit(":")))
        )
    )
    .withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
    .withColumn("time", date_format(to_timestamp(col("time"), "HH-mm-ss"), "HH:mm:ss"))
    .withColumn("ingested_at", current_timestamp())
    .withColumn("source_file", lit(os.path.basename(csv_path)))
    .withColumn("batch_id", lit(str(uuid.uuid4())))
)
df_bronze.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|status|responsible_department|wind_speed|wind_direction|energy_produced|        measured_at|         ingested_at|         source_file|            batch_id|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|         6481|2024-06-16|00:00:00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|Online|            Operations|  17.49365|             W|      1878.9614|2024-06-16 00:00:00|2026-05-03 21:46:...|20240616_wind_pow...|d0

In [9]:
# Sélection des colonnes dans l'ordre attendu
columns_table = [
    "production_id", "date", "time", "turbine_name", "capacity", "location_name", "latitude", "longitude", "region", "status", "responsible_department", "wind_speed", "wind_direction", "energy_produced", "measured_at", "ingested_at", "source_file", "batch_id"
]
df_bronze = df_bronze.select(*columns_table)
df_bronze.show(5)

+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|production_id|      date|    time|turbine_name|capacity|location_name|latitude|longitude|  region|status|responsible_department|wind_speed|wind_direction|energy_produced|        measured_at|         ingested_at|         source_file|            batch_id|
+-------------+----------+--------+------------+--------+-------------+--------+---------+--------+------+----------------------+----------+--------------+---------------+-------------------+--------------------+--------------------+--------------------+
|         6481|2024-06-16|00:00:00|   Turbine A|    2200|   Location 1| 34.0522|-118.2437|Region A|Online|            Operations|  17.49365|             W|      1878.9614|2024-06-16 00:00:00|2026-05-03 21:46:...|20240616_wind_pow...|d0

In [10]:
# Création du schéma et de la table dans PostgreSQL (si besoin)
try:
    conn = psycopg.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD
    )
    with conn:
        with conn.cursor() as cur:
            cur.execute("CREATE SCHEMA IF NOT EXISTS bronze")
            cur.execute("DROP TABLE IF EXISTS bronze.windturbinepower_raw")
            cur.execute("""
                CREATE TABLE IF NOT EXISTS bronze.windturbinepower_raw (
                    production_id INTEGER,
                    date DATE,
                    time TIME,
                    turbine_name VARCHAR(100),
                    capacity INTEGER,
                    location_name VARCHAR(100),
                    latitude DOUBLE PRECISION,
                    longitude DOUBLE PRECISION,
                    region VARCHAR(100),
                    status VARCHAR(50),
                    responsible_department VARCHAR(100),
                    wind_speed FLOAT,
                    wind_direction VARCHAR(10),
                    energy_produced FLOAT,
                    measured_at TIMESTAMP,
                    ingested_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                    source_file VARCHAR(255),
                    batch_id VARCHAR(100)
                )
            """)
            cur.execute("CREATE INDEX IF NOT EXISTS idx_windturbinepower_raw_turbine_time ON bronze.windturbinepower_raw(turbine_name, measured_at)")
            cur.execute("CREATE INDEX IF NOT EXISTS idx_windturbinepower_raw_location ON bronze.windturbinepower_raw(latitude, longitude)")
            conn.commit()
    print("Schéma et table créés avec succès.")
except Exception as e:
    print(f"Erreur lors de la création du schéma/table : {e}")
    raise

Schéma et table créés avec succès.


In [11]:
# Insertion des données dans PostgreSQL via JDBC
df_bronze.write \
    .format("jdbc") \
    .option("url", JDBC_URL) \
    .option("dbtable", "bronze.windturbinepower_raw") \
    .option("stringtype", "unspecified") \
    .options(**JDBC_PROPS) \
    .mode("append") \
    .save()
print(f"{df_bronze.count()} lignes insérées dans bronze.windturbinepower_raw")

26/05/03 21:46:03 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


432 lignes insérées dans bronze.windturbinepower_raw


In [12]:
# Arrêt de la session Spark
spark.stop()